In [1]:
import os 
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()
llm=init_chat_model(
    model="nvidia/nemotron-3-super-120b-a12b",
    model_provider="nvidia",
    api_key=os.getenv("NVIDIA_API_KEY"),
    temperature=0.2
)
print(llm.invoke("say hi").content)

Hi! 👋 How can I help you today?


# First Chain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
topic=input("user input")

prompt_template=ChatPromptTemplate.from_messages([
    ("system":"you are helpful assitance"),
    ("user":"write a fun fact about {topic}")
])


# String Parser

In [2]:
from langchain_core.output_parsers import StrOutputParser
str_parser=StrOutputParser()

# manual invokiation

In [3]:
template=prompt_template.invoke("input":"tell me a joke")
res =llm.invoke(template)
#its bad not use this method

SyntaxError: invalid syntax (738126788.py, line 1)

# CHain invokiation

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

topic = input("User input: ")

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Write a fun fact about {topic}.")
])

str_parser = StrOutputParser()

chain = prompt_template | llm | str_parser

result = chain.invoke({"topic": topic})

print(result)

Here's a fun fact about Jordan's capital, Amman:

**Amman is known as the "City of Seven Hills" (like ancient Rome!)**  
Built across seven jabals (hills), Amman’s historic core sprawls over Jabal al-Qal’a (where the Citadel sits), Jabal al-Ashrafiyah, Jabal al-Lweibdeh, and others—giving it a charmingly layered, hilly landscape perfect for exploring on foot (or discovering hidden cafes with great views!). This geography isn’t just pretty—it’s shaped the city’s growth for thousands of years, from its ancient roots as Rabbath-Ammon to the bustling modern metropolis it is today. 🏔️✨

*Bonus fun: The seven hills nickname is so iconic that locals often just say "saba jabal" (Seven Hills) when referring to the heart of the city!*


In [5]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate


# 1. Define the output schema
class FunFact(BaseModel):
    topic: str = Field(description="The topic of the fun fact")
    fun_fact: str = Field(description="An interesting and accurate fact about the topic")


# 2. Create the structured-output model
structured_llm = llm.with_structured_output(FunFact)


# 3. Get user input
topic = input("User input: ")


# 4. Create the prompt template
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Provide one interesting, accurate fun fact."),
    ("human", "Write a fun fact about {topic}.")
])


# 5. Build the chain
chain = prompt_template | structured_llm


# 6. Invoke the chain
result = chain.invoke({"topic": topic})


# 7. Display the result
print(result)

print("\nTopic:", result.topic)
print("Fun fact:", result.fun_fact)

# Convert the Pydantic object to a dictionary
print(result.model_dump())

topic='capital of Jordan' fun_fact='Amman, the capital of Jordan, is built on seven hills (Jabal) and has been continuously inhabited for over 8,000 years—making it one of the world’s oldest cities.'

Topic: capital of Jordan
Fun fact: Amman, the capital of Jordan, is built on seven hills (Jabal) and has been continuously inhabited for over 8,000 years—making it one of the world’s oldest cities.
{'topic': 'capital of Jordan', 'fun_fact': 'Amman, the capital of Jordan, is built on seven hills (Jabal) and has been continuously inhabited for over 8,000 years—making it one of the world’s oldest cities.'}


# 2:43 video reach

In [7]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda


# 1. Define the Pydantic schema
class FunFact(BaseModel):
    topic: str = Field(description="The topic of the fun fact")
    fun_fact: str = Field(description="An interesting fact about the topic")


# 2. Configure structured output
structured_llm = llm.with_structured_output(FunFact)


# 3. Create the prompt
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Provide one interesting fact."),
    ("human", "Write a fun fact about {topic}.")
])


# 4. Build the chain
chain = (
    prompt_template
    | structured_llm
    | RunnableLambda(lambda result: result.model_dump_json(indent=2))
    | StrOutputParser()
)


# 5. Get user input and invoke
topic = input("User input: ")

result = chain.invoke({"topic": topic})

print(result)

{
  "topic": "capital of Jordan",
  "fun_fact": "Amman, the capital of Jordan, is built on seven hills—just like ancient Rome—giving it a historic nickname \"the City of Seven Hills.\""
}
